# Generative Models

One of the most commonly used generative models is the variational autoencoder (VAE).

In this notebook, we study the application of the VAE on the Fashion MNIST dataset.
Fashion MNIST is nicer than MNIST for this example because the category boundaries are less distinct: things like "pullover" vs. "coat" vs. "shirt" blend into each other in latent space, which is more interesting for our exploration of the latent space.


## Visualizing the data

For once, let's show the entire dataset so that we have a good idea of the data available.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

We'll load the training dataset just to have a peek.

Do you have any idea of how many images to expect? 30, 50, 200?

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

# Define a transformation to convert images to tensors
transform = transforms.Compose([
    transforms.ToTensor()
])

# Load the Fashion MNIST training dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Create a DataLoader to easily get batches of images
batch_size = 200 # We want to display ROWS * COLS images, so let's set batch size accordingly
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

print(f"Number of training samples: {len(train_dataset)}")

# Get one batch of training images
dataiter = iter(train_loader)
images, labels = next(dataiter)

print(f"Shape of image batch: {images.shape}")
print(f"Shape of label batch: {labels.shape}")

We see there are a lot of images in the dataset!

Let's go ahead and plot them. Remember that these are shuffled input vectors from the `train_dataset`.

In [ ]:
import matplotlib.pyplot as plt

# Iterate through the first ROWS * COLS images in the batch
ROWS = 10
COLS = 20

plt.figure(figsize=(COLS * 1.2, ROWS * 1.2))

for i in range(ROWS * COLS):
    plt.subplot(ROWS, COLS, i + 1)
    # For grayscale, we can squeeze the channel dimension.
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.axis('off')
    plt.title(train_dataset.classes[labels[i]], fontsize=12)

plt.suptitle('Fashion MNIST Dataset Samples (PyTorch)', fontsize=16)
plt.show()

In fact, we cannot easily visualize the entire dataset! The dataset is composed of many different images but a smaller number of possible labels.

For example, we see many different images labeled as "dress" without any more specific classification.

We should keep this in mind when using the dataset for classification.

## Architecture

We construct a VAE in the encoder-decoder architecture. This should be familiar from our autoencoder examples.

- Can you see the size of the latent space for this VAE?
- Why is it important that the last layer of the decoder be `Sigmoid` and not `ReLU`?
- How many hidden layers are in the VAE?

In [ ]:
import torch.nn as nn
from collections import namedtuple

VAEOutput = namedtuple("VAEOutput",
                       ["output", "codings_mean", "codings_logvar"])

class VAE(nn.Module):
    def __init__(self, codings_dim=32):
        super(VAE, self).__init__()
        self.codings_dim = codings_dim
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 128), nn.ReLU(),
            nn.Linear(128, 2 * codings_dim))  # output both the mean and logvar
        self.decoder = nn.Sequential(
            nn.Linear(codings_dim, 128), nn.ReLU(),
            nn.Linear(128, 1 * 28 * 28), nn.Sigmoid(),
            nn.Unflatten(dim=1, unflattened_size=(1, 28, 28)))

    def encode(self, X):
        return self.encoder(X).chunk(2, dim=-1)  # returns (mean, logvar)

    def sample_codings(self, codings_mean, codings_logvar):
        codings_std = torch.exp(0.5 * codings_logvar)
        noise = torch.randn_like(codings_std)
        return codings_mean + noise * codings_std

    def decode(self, Z):
        return self.decoder(Z)

    def forward(self, X):
        codings_mean, codings_logvar = self.encode(X)
        codings = self.sample_codings(codings_mean, codings_logvar)
        output = self.decode(codings)
        return VAEOutput(output, codings_mean, codings_logvar)


### Training the VAE

The VAE loss function uses the KL distance in addition to the MSE.

What happens if the `kl_weight` is adjusted to a different number? (Form your hypothesis first and then test it.)

In [ ]:
import torch.nn.functional as F
def vae_loss(y_pred, y_target, kl_weight=1.0):
    output, mean, logvar = y_pred
    kl_div = -0.5 * torch.sum(1 + logvar - logvar.exp() - mean.square(), dim=-1)
    return F.mse_loss(output, y_target) + kl_weight * kl_div.mean() / 784

### Training function

The training function itself is very similar to past training functions. In fact, there is no reason to change anything, as long as we can pass in our model and loss function definitions.

In [ ]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            if isinstance(y_pred, tuple):
                y_pred = y_pred.output
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            if isinstance(y_pred, tuple):
                y_pred = y_pred.output
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.3f}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.3}, "
              f"valid metric: {history['valid_metrics'][-1]:.3}")
    return history


### Set up training, validation, and test data

Before training, we have to go back and set up the DataLoaders for training instead of for simple visualization.

In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=transform)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=transform)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])

class AutoencoderDataset(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x, _ = self.base_dataset[idx]
        return x, x

train_loader = DataLoader(AutoencoderDataset(train_data), batch_size=32,
                          shuffle=True)
valid_loader = DataLoader(AutoencoderDataset(valid_data), batch_size=32)
test_loader = DataLoader(AutoencoderDataset(test_data), batch_size=32)

## Training cycles

Here is the training itself, adjusting the VAE model `vae` with the training dataset from the `train_loader`.

In [ ]:
!pip install torchmetrics
import torchmetrics
torch.manual_seed(42)
vae = VAE().to(device)
optimizer = torch.optim.NAdam(vae.parameters(), lr=1e-3)
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train(vae, optimizer, vae_loss, rmse, train_loader, valid_loader,
                n_epochs=20)

### Visualize results

We can run the validation set input features through the trained model and see how the autoencoder has succeeded in reproducing the input at its output, at least for the first `n_images`.

In [ ]:
def plot_image(image):
    plt.imshow(image.permute(1, 2, 0).cpu(), cmap="binary")
    plt.axis("off")

def plot_reconstructions(model, images, n_images=5):
    images = images[:n_images]
    with torch.no_grad():
        y_pred = model(images.to(device))
    if isinstance(y_pred, tuple):
        y_pred = y_pred.output
    fig = plt.figure(figsize=(len(images) * 1.5, 3))
    for idx in range(len(images)):
        plt.subplot(2, len(images), 1 + idx)
        plot_image(images[idx])
        plt.subplot(2, len(images), 1 + len(images) + idx)
        plot_image(y_pred[idx])

X_valid = torch.stack([x for x, _ in valid_data])
plot_reconstructions(vae, X_valid)
plt.show()

Why are the images so fuzzy? (*Hint*: what is the dimension of the latent space of the autoencoder, compared to the number of pixels in each input image?)

Try changing `codings_dim` to see how things can be made better or worse.

## Generating new images consistent with the dataset

Here is where the "generative" part happens.

The latent space (latent dimension) is supposed to hold the essential information for each image. If we can generate some random numbers and inject them into the latent space, then we can use the VAE decoder to "decode" this random input into Fashion MNIST images.

In this way, we *generate* new Fashion MNIST images from a tiny model. Let's see if it works!

In [ ]:
torch.manual_seed(42)  # extra code – ensures reproducibility

vae.eval()
# Slip some random numbers into the VAE latent space
codings = torch.randn(3 * 7, vae.codings_dim, device=device)
with torch.no_grad():
    images = vae.decode(codings)

def plot_multiple_images(images, n_cols=None):
    n_cols = n_cols or len(images)
    n_rows = (len(images) - 1) // n_cols + 1
    plt.figure(figsize=(n_cols, n_rows))
    for index, image in enumerate(images):
        plt.subplot(n_rows, n_cols, index + 1)
        plot_image(image)

plot_multiple_images(images, 7)
plt.show()

You can try different random seeds to confirm that the VAE is generating new images.

## Interpolation between images

We can morph between two images by interpolating between two random representations in the latent space.

We step linearly between the one random representation and the second representation, then decode the linear interpretations from the latent space into output images.

In [ ]:
torch.manual_seed(111)  # ensure reproducibility

codings = torch.randn(2, vae.codings_dim)  # start and end codings for morphing
n_images = 7
weights = torch.linspace(0, 1, n_images).view(n_images, 1)
codings = torch.lerp(codings[0], codings[1], weights)  # linear interpolation
with torch.no_grad():
    images = vae.decode(codings.to(device))

plot_multiple_images(images)
plt.show()

### Interpretation

Does it seem like the interpolated images, coming from the sum of two random vectors, are sloppier than the images we generate above with single random vectors?

Why might this be the case? How could you check it?